# Planejamento de rede mesh com inclusão de repetidores

SDK 0.1.60 ou superior e microserviço atualizado. O projeto começa com dois acessos fixos (Guaira e Italia) e três terminais (Copel, Woodland e Cemiterio). Os postes ficam separados em `poles.csv`. A inclusão de repetidores é uma etapa específica; depois dela, seguimos as operações comuns do planejador de grafos.

In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1800)
project = PlanningProject("../projects/example_5_Mesh_network")
scenario = client.open_project(project, planner="mesh")

## 1 Adicionar repetidores

Esta chamada executa o único laço especializado. Para cada terminal, compara conexões com pontos servidos próximos usando o planejador de enlaces, que pode acrescentar repetidores em postes disponíveis. Os repetidores incorporados podem atender as rodadas seguintes. A métrica necessária às buscas é calculada dentro dos estudos de enlace.

Ao terminar, o cenário ampliado contém os terminais e os repetidores propostos. Somente os acessos originais são fixos; a conectividade temporária das rodadas não cria novas saídas. As entradas originais são preservadas e o cenário ampliado é salvo em `mesh_expanded`.

In [ ]:
repeaters = scenario.add_repeaters(geo_base_url="http://features:8080")

## Inspecionar o primeiro estudo de inclusão

Os pontos representam os postes disponíveis no corredor e os extremos. As linhas representam os enlaces efetivamente avaliados. Um poste isolado pode não ter sido explorado. O primeiro estudo pode ser uma alternativa diferente da escolhida; a configuração ampliada reúne apenas os repetidores das propostas aceitas.

In [ ]:
candidate_study = scenario.show_mesh_study(1, attempt=1, kind="candidate")

In [ ]:
link_study = scenario.show_mesh_study(1, attempt=1, kind="final")

## 2 Construir o grafo candidato

Agora usamos o GraphPlanner sobre os rádios originais e os repetidores propostos. A operação constrói alternativas de conexão pelas regras de conectividade e não consulta features, não procura novos postes e não acrescenta equipamentos.

In [ ]:
candidates = scenario.build_candidate_edges()

## 3 Calcular as métricas

Esta operação apenas avalia os enlaces do grafo candidato. O conjunto de rádios e enlaces permanece o mesmo. A extração reutiliza as features já salvas durante a inclusão dos repetidores. No mesh, os dois sentidos são avaliados e o maior custo é utilizado.

In [ ]:
metrics = scenario.compute_metrics(geo_base_url="http://features:8080")

In [ ]:
candidate_network = scenario.show_network(kind="candidate")

## 4 Selecionar os caminhos

A operação comum de solução seleciona os caminhos até os acessos fixos originais. Não acrescenta nem retira rádios. O SDK utiliza o limite por segmento informado no estudo mesh; ele pode ser informado explicitamente com `max_edge_metric`. Candidatos fora do limite continuam disponíveis para inspeção, mas não entram nos caminhos selecionados.

In [ ]:
solution = scenario.solve()

## 5 Retirar os repetidores sem uso

Depois da seleção, esta operação retira somente repetidores propostos que não aparecem nos caminhos dos terminais obrigatórios. Os terminais e acessos originais são preservados. Não são feitas novas extrações de propagação.

In [ ]:
pruning = scenario.remove_unused_repeaters()

## 6 Salvar e inspecionar a rede final

O resultado completo fica em `result.json`, os estados da inclusão em `mesh_iterations` e o projeto final em `mesh_solution`. O `scenario.toml` de saída contém as instâncias completas; o `nodes.csv` correspondente mantém um inventário por site quando os perfis permitem essa representação. Os arquivos de entrada originais permanecem intactos. A visualização destaca os terminais sem caminho, se houver.

In [ ]:
saved_result = scenario.save_result()
result = scenario.load_result()

In [ ]:
solution_network = scenario.show_network()